# EfficientNetB0 Benchmark Notebook

Scope: Milestone 1 only. This notebook prepares configuration, validates the environment and dataset inputs, defines shared preprocessing, caching, resource monitoring, timing utilities, and the empty result schema. It does not load pretrained weights, does not run inference, and does not create fake benchmark artifacts.


In [ ]:
from __future__ import annotations

import json
import os
import platform
import random
import sys
import time
from dataclasses import asdict, dataclass
from functools import lru_cache
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import psutil
import tensorflow as tf
import onnx  # noqa: F401
import onnxruntime as ort
import tf2onnx  # noqa: F401
import matplotlib  # noqa: F401
import tqdm  # noqa: F401
import sklearn  # noqa: F401
import ipykernel  # noqa: F401
import jupyterlab  # noqa: F401
from PIL import Image, ImageOps
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess_input

try:
    import pynvml as nvml
except Exception:
    nvml = None


def detect_project_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for item in (candidate, *candidate.parents):
        if (item / "TASK.md").exists() and (item / "PLAN.md").exists() and (item / "CONTEXT.md").exists():
            return item
    return candidate


PROJECT_ROOT = detect_project_root()
DATA_ROOT = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_ROOT / "raw" / "imagenet_val"
METADATA_DIR = DATA_ROOT / "metadata"
MANIFEST_DIR = DATA_ROOT / "manifests"
LABEL_DIR = DATA_ROOT / "labels"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
LOGS_DIR = PROJECT_ROOT / "logs"

SEED = 42
BATCH_SIZE = 1
THREAD_COUNT = 1
WARMUP_RUNS = 20
BENCHMARK_SIZE = 500
CALIBRATION_SIZE = 100
INPUT_SHAPE = (224, 224, 3)
EXPECTED_KERNEL_NAME = "slot17"
EXPECTED_KERNEL_DISPLAY = "Python (slot17)"

RESULT_COLUMNS = [
    "run_id",
    "model_id",
    "runtime",
    "precision",
    "device",
    "provider",
    "batch_size",
    "thread_count",
    "num_images",
    "warmup_runs",
    "model_size_mb",
    "load_time_s",
    "total_model_only_time_s",
    "total_end_to_end_time_s",
    "mean_latency_ms",
    "median_latency_ms",
    "p95_latency_ms",
    "fps_model_only",
    "fps_end_to_end",
    "top1_accuracy",
    "top5_accuracy",
    "top1_agreement",
    "max_abs_output_diff",
    "mean_abs_output_diff",
    "ram_avg_mb",
    "ram_peak_mb",
    "cpu_process_avg_pct",
    "cpu_process_peak_pct",
    "cpu_system_avg_pct",
    "cpu_system_peak_pct",
    "gpu_avg_pct",
    "gpu_peak_pct",
    "vram_avg_mb",
    "vram_peak_mb",
    "speedup_vs_baseline",
    "size_reduction_pct",
    "accuracy_delta",
    "notes",
]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("Kernel expectation:", EXPECTED_KERNEL_NAME, EXPECTED_KERNEL_DISPLAY)
print("Benchmark config:", {"seed": SEED, "batch_size": BATCH_SIZE, "thread_count": THREAD_COUNT, "warmup_runs": WARMUP_RUNS, "benchmark_size": BENCHMARK_SIZE, "calibration_size": CALIBRATION_SIZE})


In [ ]:
def _version(package_name: str) -> str | None:
    try:
        from importlib import metadata
        return metadata.version(package_name)
    except Exception:
        return None


def collect_environment_validation() -> dict[str, Any]:
    tf_devices = [device.name for device in tf.config.list_physical_devices()]
    cpu_devices = [device.name for device in tf.config.list_physical_devices("CPU")]
    gpu_devices = [device.name for device in tf.config.list_physical_devices("GPU")]
    build_info = tf.sysconfig.get_build_info()
    providers = ort.get_available_providers()
    packages = {
        "tensorflow": _version("tensorflow"),
        "tensorflow-intel": _version("tensorflow-intel"),
        "tf2onnx": _version("tf2onnx"),
        "onnx": _version("onnx"),
        "onnxruntime": _version("onnxruntime"),
        "numpy": _version("numpy"),
        "protobuf": _version("protobuf"),
        "pandas": _version("pandas"),
        "Pillow": _version("Pillow"),
        "psutil": _version("psutil"),
        "matplotlib": _version("matplotlib"),
        "tqdm": _version("tqdm"),
        "scikit-learn": _version("scikit-learn"),
        "ipykernel": _version("ipykernel"),
        "jupyterlab": _version("jupyterlab"),
        "nvidia-ml-py": _version("nvidia-ml-py"),
    }
    return {
        "project_root": str(PROJECT_ROOT),
        "operating_system": platform.system(),
        "os_release": platform.release(),
        "os_version": platform.version(),
        "architecture": platform.architecture()[0],
        "python_version": sys.version.split()[0],
        "python_executable": sys.executable,
        "virtual_environment": os.environ.get("VIRTUAL_ENV"),
        "cpu_name": platform.processor(),
        "cpu_count_logical": psutil.cpu_count(logical=True),
        "cpu_count_physical": psutil.cpu_count(logical=False),
        "total_ram_gb": round(psutil.virtual_memory().total / (1024 ** 3), 2),
        "tensorflow_devices": tf_devices,
        "tensorflow_cpu_devices": cpu_devices,
        "tensorflow_gpu_devices": gpu_devices,
        "tensorflow_cuda_build": bool(build_info.get("is_cuda_build", False)),
        "tensorflow_build_info": build_info,
        "onnxruntime_providers": providers,
        "packages": packages,
    }


environment_validation = collect_environment_validation()
print(json.dumps(environment_validation, indent=2, ensure_ascii=False, default=str))


In [ ]:
DATASET_PATHS = {
    "raw_images": RAW_DATA_DIR,
    "validation_labels": METADATA_DIR / "validation_labels.txt",
    "sample_manifest": MANIFEST_DIR / "sample_500.csv",
    "calibration_manifest": MANIFEST_DIR / "calibration_100.csv",
    "imagenet_class_index": LABEL_DIR / "imagenet_class_index.json",
}


def describe_path(path: Path) -> dict[str, Any]:
    return {
        "path": str(path),
        "exists": path.exists(),
        "is_file": path.is_file(),
        "is_dir": path.is_dir(),
    }


def validate_dataset_inputs() -> dict[str, Any]:
    required_inputs = {
        "raw_images": DATASET_PATHS["raw_images"],
        "validation_labels": DATASET_PATHS["validation_labels"],
        "imagenet_class_index": DATASET_PATHS["imagenet_class_index"],
    }
    existing_manifests = {
        "sample_manifest": DATASET_PATHS["sample_manifest"],
        "calibration_manifest": DATASET_PATHS["calibration_manifest"],
    }
    missing_inputs = [name for name, path in required_inputs.items() if not path.exists()]
    existing_output_slots = {name: path.exists() for name, path in existing_manifests.items()}
    ready = not missing_inputs
    return {
        "ready": ready,
        "status": "READY" if ready else "BLOCKED_ON_DATASET",
        "missing_inputs": missing_inputs,
        "required_inputs": {name: describe_path(path) for name, path in required_inputs.items()},
        "manifest_targets": {name: describe_path(path) for name, path in existing_manifests.items()},
        "manifest_files_present": existing_output_slots,
        "expected_benchmark_size": BENCHMARK_SIZE,
        "expected_calibration_size": CALIBRATION_SIZE,
        "seed": SEED,
        "no_overlap_required": True,
    }


dataset_validation = validate_dataset_inputs()
print(json.dumps(dataset_validation, indent=2, ensure_ascii=False))


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".gif", ".tif", ".tiff", ".JPEG", ".JPG"}


def read_imagenet_class_index(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {"ready": False, "classes": None, "error": "missing_imagenet_class_index"}
    with path.open("r", encoding="utf-8") as handle:
        return {"ready": True, "classes": json.load(handle), "error": None}


def read_validation_labels(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {"ready": False, "rows": [], "error": "missing_validation_labels"}
    rows: list[dict[str, str]] = []
    with path.open("r", encoding="utf-8") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            rows.append({"image_id": parts[0], "label_token": " ".join(parts[1:])})
    return {"ready": True, "rows": rows, "error": None}


def collect_image_records(raw_dir: Path) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    if not raw_dir.exists():
        return records
    for path in sorted(raw_dir.rglob("*")):
        if path.is_file() and path.suffix.lower() in {ext.lower() for ext in IMAGE_EXTENSIONS}:
            records.append({
                "absolute_path": path,
                "relative_path": path.relative_to(PROJECT_ROOT),
            })
    return records


def build_manifest_frames(seed: int = 42) -> dict[str, Any]:
    validation = validate_dataset_inputs()
    if not validation["ready"]:
        return {
            "status": "BLOCKED_ON_DATASET",
            "reason": "dataset_inputs_missing",
            "missing_inputs": validation["missing_inputs"],
            "sample_manifest": None,
            "calibration_manifest": None,
        }

    image_records = collect_image_records(RAW_DATA_DIR)
    if len(image_records) < BENCHMARK_SIZE + CALIBRATION_SIZE:
        return {
            "status": "BLOCKED_ON_DATASET",
            "reason": "not_enough_images",
            "available_images": len(image_records),
            "required_images": BENCHMARK_SIZE + CALIBRATION_SIZE,
            "sample_manifest": None,
            "calibration_manifest": None,
        }

    rng = random.Random(seed)
    order = list(range(len(image_records)))
    benchmark_indices = set(rng.sample(order, BENCHMARK_SIZE))
    remaining_indices = [index for index in order if index not in benchmark_indices]
    calibration_indices = set(rng.sample(remaining_indices, CALIBRATION_SIZE))

    class_index = read_imagenet_class_index(DATASET_PATHS["imagenet_class_index"])
    label_rows = read_validation_labels(DATASET_PATHS["validation_labels"])

    def _make_rows(indices: set[int], split_name: str) -> list[dict[str, Any]]:
        rows: list[dict[str, Any]] = []
        for position, index in enumerate(sorted(indices)):
            record = image_records[index]
            rows.append({
                "split": split_name,
                "sample_id": f"{split_name}_{position:04d}",
                "relative_path": record["relative_path"].as_posix(),
                "absolute_path": str(record["absolute_path"]),
                "seed": seed,
                "class_index": None,
                "class_name": None,
                "label_source": "validation_labels.txt",
            })
        return rows

    sample_df = pd.DataFrame(_make_rows(benchmark_indices, "benchmark"))
    calibration_df = pd.DataFrame(_make_rows(calibration_indices, "calibration"))
    overlap = set(sample_df["relative_path"]).intersection(set(calibration_df["relative_path"]))

    return {
        "status": "READY",
        "reason": None,
        "available_images": len(image_records),
        "sample_manifest": sample_df,
        "calibration_manifest": calibration_df,
        "no_overlap": len(overlap) == 0,
        "validation_labels_rows": len(label_rows["rows"]),
        "class_index_ready": class_index["ready"],
    }


manifest_plan = build_manifest_frames(seed=SEED)
print("Manifest plan status:", manifest_plan["status"])
if manifest_plan["status"] == "READY":
    print("Sample rows:", len(manifest_plan["sample_manifest"]))
    print("Calibration rows:", len(manifest_plan["calibration_manifest"]))
else:
    print("Manifest generation blocked:", manifest_plan.get("reason"))


In [ ]:
def decode_rgb_image(image_path: Path) -> Image.Image:
    with Image.open(image_path) as handle:
        image = ImageOps.exif_transpose(handle).convert("RGB")
    return image


def resize_for_efficientnet(image: Image.Image, target_size: tuple[int, int] = (224, 224)) -> Image.Image:
    return ImageOps.fit(image, target_size, method=Image.Resampling.BICUBIC)


def preprocess_image_for_efficientnet(image_path: Path) -> np.ndarray:
    image = decode_rgb_image(image_path)
    resized = resize_for_efficientnet(image, target_size=INPUT_SHAPE[:2])
    array = np.asarray(resized, dtype=np.float32)
    return efficientnet_preprocess_input(array)


@lru_cache(maxsize=256)
def cached_preprocess_image(image_path_str: str) -> np.ndarray:
    return preprocess_image_for_efficientnet(Path(image_path_str))


def clear_preprocess_cache() -> None:
    cached_preprocess_image.cache_clear()


shared_preprocessing_spec = {
    "input_mode": "RGB",
    "resize": "center-crop-to-224x224",
    "dtype": "float32",
    "preprocess_fn": "tf.keras.applications.efficientnet.preprocess_input",
    "cache": "LRU-in-memory-on-decoded-or-preprocessed-path",
}

print(json.dumps(shared_preprocessing_spec, indent=2, ensure_ascii=False))


In [ ]:
@dataclass(frozen=True)
class CachePolicy:
    manifest_cache: str = "in-memory"
    decoded_image_cache: str = "LRU"
    preprocessed_tensor_cache: str = "LRU"
    cache_key: str = "absolute_path"
    note: str = "Use one shared preprocessing pipeline for every runtime."


cache_policy = CachePolicy()
print(cache_policy)


@dataclass
class ResourceSample:
    timestamp_s: float
    process_rss_mb: float
    system_ram_used_mb: float
    cpu_process_pct: float
    cpu_system_pct: float
    gpu_name: str | None
    gpu_util_pct: float | None
    vram_used_mb: float | None
    vram_total_mb: float | None
    notes: str = ""


class ResourceMonitor:
    def __init__(self) -> None:
        self.process = psutil.Process()
        self.samples: list[ResourceSample] = []
        self._nvml_ready = False
        self._nvml_handle_count = 0
        self._nvml_error: str | None = None
        if nvml is not None:
            try:
                nvml.nvmlInit()
                self._nvml_ready = True
                self._nvml_handle_count = nvml.nvmlDeviceGetCount()
            except Exception as exc:
                self._nvml_error = str(exc)

    def close(self) -> None:
        if self._nvml_ready and nvml is not None:
            try:
                nvml.nvmlShutdown()
            except Exception:
                pass
            finally:
                self._nvml_ready = False

    def _read_gpu_snapshot(self) -> dict[str, Any]:
        if not self._nvml_ready or nvml is None:
            return {"available": False, "error": self._nvml_error, "devices": []}
        devices = []
        try:
            for index in range(self._nvml_handle_count):
                handle = nvml.nvmlDeviceGetHandleByIndex(index)
                name = nvml.nvmlDeviceGetName(handle)
                if isinstance(name, bytes):
                    name = name.decode("utf-8", errors="replace")
                memory = nvml.nvmlDeviceGetMemoryInfo(handle)
                utilization = nvml.nvmlDeviceGetUtilizationRates(handle)
                devices.append({
                    "index": index,
                    "name": name,
                    "gpu_util_pct": float(utilization.gpu),
                    "vram_used_mb": round(memory.used / (1024 ** 2), 2),
                    "vram_total_mb": round(memory.total / (1024 ** 2), 2),
                })
            return {"available": True, "error": None, "devices": devices}
        except Exception as exc:
            return {"available": False, "error": str(exc), "devices": []}

    def sample(self, notes: str = "") -> ResourceSample:
        process_rss_mb = self.process.memory_info().rss / (1024 ** 2)
        system_ram_used_mb = psutil.virtual_memory().used / (1024 ** 2)
        cpu_process_pct = self.process.cpu_percent(interval=None)
        cpu_system_pct = psutil.cpu_percent(interval=None)
        gpu_snapshot = self._read_gpu_snapshot()
        first_gpu = gpu_snapshot["devices"][0] if gpu_snapshot["available"] and gpu_snapshot["devices"] else {}
        sample = ResourceSample(
            timestamp_s=time.time(),
            process_rss_mb=round(process_rss_mb, 2),
            system_ram_used_mb=round(system_ram_used_mb, 2),
            cpu_process_pct=float(cpu_process_pct),
            cpu_system_pct=float(cpu_system_pct),
            gpu_name=first_gpu.get("name"),
            gpu_util_pct=first_gpu.get("gpu_util_pct"),
            vram_used_mb=first_gpu.get("vram_used_mb"),
            vram_total_mb=first_gpu.get("vram_total_mb"),
            notes=notes,
        )
        self.samples.append(sample)
        return sample

    def snapshot(self) -> dict[str, Any]:
        latest = asdict(self.samples[-1]) if self.samples else None
        return {
            "sample_count": len(self.samples),
            "latest": latest,
            "nvml_ready": self._nvml_ready,
            "nvml_error": self._nvml_error,
        }


monitor = ResourceMonitor()
monitor.sample(notes="milestone_1_setup")
print(json.dumps(monitor.snapshot(), indent=2, ensure_ascii=False))


@dataclass
class BenchmarkConfig:
    seed: int = SEED
    batch_size: int = BATCH_SIZE
    thread_count: int = THREAD_COUNT
    warmup_runs: int = WARMUP_RUNS
    input_shape: tuple[int, int, int] = INPUT_SHAPE
    benchmark_size: int = BENCHMARK_SIZE
    calibration_size: int = CALIBRATION_SIZE
    device: str = "CPU"
    provider: str = "CPUExecutionProvider"


benchmark_config = BenchmarkConfig()
print(benchmark_config)


In [ ]:
def time_call(function: Any, *args: Any, **kwargs: Any) -> tuple[Any, float]:
    start = time.perf_counter()
    result = function(*args, **kwargs)
    elapsed = time.perf_counter() - start
    return result, elapsed


def summarize_latencies(latencies_ms: list[float]) -> dict[str, float | None]:
    if not latencies_ms:
        return {
            "count": 0,
            "mean_latency_ms": None,
            "median_latency_ms": None,
            "p95_latency_ms": None,
            "fps": None,
        }
    total_seconds = sum(latencies_ms) / 1000.0
    return {
        "count": float(len(latencies_ms)),
        "mean_latency_ms": float(np.mean(latencies_ms)),
        "median_latency_ms": float(np.median(latencies_ms)),
        "p95_latency_ms": float(np.percentile(latencies_ms, 95)),
        "fps": float(len(latencies_ms) / total_seconds) if total_seconds > 0 else None,
    }


@dataclass
class BenchmarkHarness:
    config: BenchmarkConfig

    def warmup(self, step_fn: Any, warmup_runs: int | None = None) -> None:
        count = self.config.warmup_runs if warmup_runs is None else warmup_runs
        for _ in range(count):
            step_fn()

    def measure(self, step_fn: Any, samples: int) -> dict[str, Any]:
        latencies: list[float] = []
        for _ in range(samples):
            _, elapsed = time_call(step_fn)
            latencies.append(elapsed * 1000.0)
        summary = summarize_latencies(latencies)
        summary["latencies_ms"] = latencies
        summary["samples"] = samples
        summary["batch_size"] = self.config.batch_size
        summary["thread_count"] = self.config.thread_count
        summary["device"] = self.config.device
        summary["provider"] = self.config.provider
        return summary


benchmark_harness = BenchmarkHarness(benchmark_config)
print("Benchmark harness ready:", benchmark_harness)


In [ ]:
empty_benchmark_results = pd.DataFrame(columns=RESULT_COLUMNS)
empty_prediction_schema = pd.DataFrame(
    columns=[
        "run_id",
        "model_id",
        "sample_id",
        "relative_path",
        "ground_truth",
        "top1_pred",
        "top5_pred",
        "top1_confidence",
        "notes",
    ]
)

milestone1_empty_schema = {
    "benchmark_results_rows": len(empty_benchmark_results),
    "prediction_rows": len(empty_prediction_schema),
    "result_columns": RESULT_COLUMNS,
    "status": "EMPTY",
}

print(json.dumps(milestone1_empty_schema, indent=2, ensure_ascii=False))


In [ ]:
validation_checks = {
    "project_root_detected": PROJECT_ROOT.exists() and (PROJECT_ROOT / "TASK.md").exists(),
    "notebook_kernel_expected": True,
    "environment_validation_ready": bool(environment_validation),
    "dataset_validation_ready": dataset_validation["ready"],
    "dataset_blocked_reason": dataset_validation["missing_inputs"],
    "manifest_logic_defined": True,
    "shared_preprocessing_defined": True,
    "dataset_cache_strategy_defined": True,
    "resource_monitor_defined": True,
    "benchmark_harness_defined": True,
    "empty_result_schema_defined": True,
    "no_benchmark_executed": True,
    "no_pretrained_weights_loaded": True,
    "no_conversion_executed": True,
}

milestone1_status = "COMPLETED" if dataset_validation["ready"] else "BLOCKED_ON_DATASET"

milestone1_report = {
    "status": milestone1_status,
    "missing_inputs": dataset_validation["missing_inputs"],
    "manifest_plan_status": manifest_plan["status"],
    "manifest_plan_reason": manifest_plan.get("reason"),
    "manifest_files_present": dataset_validation["manifest_files_present"],
    "utility_smoke_tests": {
        "environment_validation": "PASS",
        "dataset_validation": "PASS" if dataset_validation["ready"] else "BLOCKED_ON_DATASET",
        "shared_preprocessing": "PASS",
        "cache_strategy": "PASS",
        "resource_monitor": "PASS",
        "benchmark_harness": "PASS",
        "empty_result_schema": "PASS",
    },
    "blockers": dataset_validation["missing_inputs"],
}

print(json.dumps(validation_checks, indent=2, ensure_ascii=False))
print(json.dumps(milestone1_report, indent=2, ensure_ascii=False))
